# Linear probes for the belief state (Section 5)

**Paper section.** Section 5 (Linearly probing the belief state geometry) with Figures 3, 4, and 5 and the middle panel of Figure 1 (b), Appendix D with Figures 11 to 15, and Appendices I, J, K, and L with Figures 29 to 52.

**Claim.** In the paper's words, "Belief states are linearly decodable from residual stream activations", the activations "carry HMM-specific belief state information", they "capture more than order-one belief state structure", and "Belief state decodability cannot be explained by next-token probability or log next-token probability representations."

**Experiment.** The notebook draws five probe experiments, all produced by `scripts/run_probes.sh`. Linear probes with bias map the late-window activations (positions 15,000 onward) to the belief state at every layer, with shuffled and random controls. Transfer probes decode every parametrization's beliefs from one parametrization's activations and are compared with their ground-truth counterpart. k-suffix probes decode the full HMM's k-suffix beliefs from activations on full-HMM, 1-HMM, and 0-HMM sequences. NTP and log-NTP probes are compared with the model-free NTP-to-belief baselines. Early-context probes are fit within the first 5,000 tokens and evaluated on the first 250.

**Saved Outputs.** In `figures/`: `r2_pooled.pdf`, `fig1_r2_wing.svg`, the five `geometry_grid_<condition>_pooled_qwen35_9b.pdf`, `r2_controls_<model>.pdf`, `within_r2.pdf`, `within_r2_<model>.pdf`, `redr2.pdf`, `rep_gap_k20_<model>.pdf`, `rep_gap_theory_seed_<model>.pdf`, `obsprob.pdf`, `ntp_gap_<model>.pdf`, `log_ntp_gap_<model>.pdf`, and `ntp_gap_theory_seed_<surrogate>_<model>.pdf`. In `results/`, on first use, the caches `theory_gap_1hmm_seedlevel.csv` and `theory_gap_seedlevel.csv`.

**How the notebook works.** The first code cell sets the paths and loads `results/r2_<model>.csv` for every model. The following cells read the geometry files, the NTP, k-suffix, and transfer probe results, and the early-context files, compute the summary statistics quoted in the paper, and write the figures to `figures/`. The theoretical counterparts of Appendices K and L are recomputed from the HMM definitions when their cache files are absent.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import os, sys
from matplotlib.lines import Line2D
from scipy.stats import pearsonr, spearmanr
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize, LinearSegmentedColormap
from matplotlib.ticker import MultipleLocator

# ═══════════════════════════════════════════════════════════
# CONFIG — change these paths, everything else follows
# ═══════════════════════════════════════════════════════════
RESULTS_DIR = '../results'
PLOT_DIR = '../figures'
os.makedirs(PLOT_DIR, exist_ok=True)

MODEL_KEYS = ['qwen35_9b', 'qwen35_4b', 'llama_31_8b', 'llama_32_3b', 'gemma_4_e4b', 'gemma_4_e2b']
MODEL_LABELS = {
    'qwen35_9b': 'Qwen 3.5 9B', 'qwen35_4b': 'Qwen 3.5 4B',
    'llama_31_8b': 'Llama 3.1 8B', 'llama_32_3b': 'Llama 3.2 3B',
    'gemma_4_e4b': 'Gemma 4 E4B', 'gemma_4_e2b': 'Gemma 4 E2B',
}

# Representative (HMM, param) for single-param plots
TARGETS = [
    ('Mess3', 'a=0.01, x=0.02'),
    # ('Arch', 'a=0.9'),
    ('Arch', 'a=0.99'),
    ('Wing', 'a=0.98, x=0.4'),
    ('Strata', 'a=0.97, t0=0.38, t1=0.54'),
]
HMM_ORDER = ['Mess3', 'Arch', 'Wing', 'Strata']
MODEL_ORDER = list(MODEL_KEYS)
HMM_COLORS = {'Mess3': 'tab:blue', 'Arch': 'tab:orange', 'Wing': 'tab:green', 'Strata': 'tab:red'}
HMMS = ['Mess3', 'Arch', 'Wing', 'Strata']

# ═══════════════════════════════════════════════════════════
# Style
# ═══════════════════════════════════════════════════════════
matplotlib.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['DejaVu Sans'],
    'font.weight': 'light',
})
sns.set_context('notebook')
tab10 = sns.color_palette('tab10')

def fmt_tick(v):
    s = f'{v:.1f}' if v == int(v) else f'{v:.2f}'
    if -1 < v < 1 and '.' in s:
        s = s.replace('0.', '.')   # 0.97 → .97  AND  -0.45 → -.45
    return s

# ═══════════════════════════════════════════════════════════
# Loaders
# ═══════════════════════════════════════════════════════════
def load_csv(prefix, model_key):
    path = os.path.join(RESULTS_DIR, f'{prefix}_{model_key}.csv')
    if not os.path.exists(path):
        print(f'  MISSING: {path}')
        return None
    return pd.read_csv(path)

def load_npz(prefix, model_key):
    path = os.path.join(RESULTS_DIR, f'{prefix}_{model_key}.npz')
    if not os.path.exists(path):
        print(f'  MISSING: {path}')
        return None
    return np.load(path, allow_pickle=True)

def harmonize_cross(df):
    if df is None: return None
    if 'source' in df.columns:
        df = df.rename(columns={'source': 'train', 'target': 'test'})
    return df

def harmonize_wing_labels(cross_df, gt_df):
    if cross_df is None or gt_df is None: return cross_df, gt_df
    for name in ['Wing']:
        c_params = set(cross_df[cross_df['hmm']==name]['train'].unique()) if name in cross_df['hmm'].values else set()
        g_params = set(gt_df[gt_df['hmm']==name]['train'].unique()) if name in gt_df['hmm'].values else set()
        if c_params and g_params and c_params != g_params:
            mask = gt_df['hmm'] == name
            for col in ['train', 'test']:
                gt_df.loc[mask, col] = (gt_df.loc[mask, col]
                    .str.replace('x=', 'ALPHA=').str.replace('y=', 'x=').str.replace('ALPHA=', 'a='))
    return cross_df, gt_df

# ═══════════════════════════════════════════════════════════
# Auto-load all R² files
# ═══════════════════════════════════════════════════════════
r2_files = {}
for mk in MODEL_KEYS:
    df = load_csv('r2', mk)
    if df is not None:
        r2_files[mk] = df
        print(f'{MODEL_LABELS[mk]}: {len(df)} rows, HMMs={sorted(df["hmm"].unique())}')
print(f'\n{len(r2_files)} models loaded')

## Peak R² across all models and parametrizations (Section 5.2)

In [ ]:
peaks = []

for model_short, df in r2_files.items():
    model_name = MODEL_LABELS.get(model_short, model_short)
    real = df[(df['target'] == 'real') & (df['hmm'] != 'Spiral')]

    for hmm_name in sorted(real['hmm'].unique()):
        for param in sorted(real[real['hmm'] == hmm_name]['param'].unique()):
            sub = real[(real['hmm'] == hmm_name) & (real['param'] == param)]
            peak_r2 = sub.groupby('layer')['R2'].mean().max()
            peaks.append({'model': model_name, 'hmm': hmm_name, 'param': param, 'peak_R2': peak_r2})

peaks_df = pd.DataFrame(peaks)
print(f'Grand mean peak R² = {peaks_df["peak_R2"].mean():.4f} ± {peaks_df["peak_R2"].std():.4f}')
print(f'Range: [{peaks_df["peak_R2"].min():.4f}, {peaks_df["peak_R2"].max():.4f}]')
print(f'N = {len(peaks_df)} (model × HMM × param)')

## Figure 3: probe R² and the recovered geometry (Qwen 3.5 9B)

In [ ]:
# Load R² + geometry — change model key as needed
MODEL = 'qwen35_9b'
r2_df = load_csv('r2', MODEL)
gd = load_npz('geom', MODEL)
print(f'R²: {len(r2_df)} rows') if r2_df is not None else None

In [ ]:
import matplotlib
matplotlib.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['DejaVu Sans'],
    'font.weight': 'light',
})

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
import os
from sklearn.decomposition import PCA
from scipy.ndimage import gaussian_filter

sns.set_context('notebook')

# ---- pooled held-out geometry: 10 sequences x 4000 held-out positions per target ----
gpool = np.load(os.path.join(RESULTS_DIR, 'geompool_all_qwen35_9b.npz'), allow_pickle=True)
gd = np.load(os.path.join(RESULTS_DIR, 'geom_qwen35_9b.npz'), allow_pickle=True)   # PCA frame for Arch (as plots/r2.pdf)
r2_df = pd.read_csv(os.path.join(RESULTS_DIR, 'r2_qwen35_9b.csv'))
GEOM_PT = 0.12

# ---- density -> opacity (colour stays the belief RGB; dense structure drawn solid, sparse points fade) ----
DENS_BINS = 150              # histogram resolution per panel
DENS_SMOOTH = 1.0            # gaussian smoothing of the histogram, in bins
DENS_GAMMA = 0.5             # opacity = (density / max density) ** gamma; smaller gamma flattens the contrast
ALPHA_MIN, ALPHA_MAX = 0.03, 0.9

def density_alpha(xy, bins=DENS_BINS):
    H, xe, ye = np.histogram2d(xy[:, 0], xy[:, 1], bins=bins)
    H = gaussian_filter(H, DENS_SMOOTH)
    ix = np.clip(np.searchsorted(xe, xy[:, 0], side='right') - 1, 0, bins - 1)
    iy = np.clip(np.searchsorted(ye, xy[:, 1], side='right') - 1, 0, bins - 1)
    d = H[ix, iy]; d = d / d.max()
    return ALPHA_MIN + (ALPHA_MAX - ALPHA_MIN) * d ** DENS_GAMMA

tab10 = sns.color_palette('tab10')
target_styles = {
    'real':    ('black',   4, 'Activations → Beliefs',          '-'),
    'shuffle': (tab10[0],  4, 'Activations → Shuffled Beliefs', '-'),
    'random':  (tab10[3],  4, 'Activations → Random Beliefs',   ':'),
}

CONN_COLOR = '0.55'

fig = plt.figure(figsize=(13, 6.5))
fig.patch.set_facecolor('white')
fig.patch.set_edgecolor('white')
fig.patch.set_linewidth(0)
outer = fig.add_gridspec(2, 2, wspace=0.2, hspace=0.3)

# Shared legend at top
from matplotlib.lines import Line2D
legend_handles = []
for target, (color, lw, label, ls) in target_styles.items():
    legend_handles.append(Line2D([], [], color=color, lw=5, ls=ls, label=label))
fig.legend(handles=legend_handles, loc='upper center', ncol=3,
           fontsize=15, frameon=False, bbox_to_anchor=(0.5, 0.025))

# First pass: create R² axes only
r2_axes = {}
for i, (hmm, param) in enumerate(TARGETS):
    row, col = divmod(i, 2)

    inner = outer[row, col].subgridspec(1, 2, width_ratios=[3, 1.2], wspace=0.05)
    inner_left = inner[0].subgridspec(2, 1, height_ratios=[70, 30], hspace=0.08)

    ax_top = fig.add_subplot(inner_left[0])
    ax_bot = fig.add_subplot(inner_left[1])

    inner_right_spec = inner[1]

    r2_axes[hmm] = (ax_top, ax_bot, inner_right_spec, i, row, col)

# Draw once to get positions
fig.canvas.draw()

for hmm, param in TARGETS:
    ax_top, ax_bot, inner_right_spec, i, row, col = r2_axes[hmm]

    renderer = fig.canvas.get_renderer()
    right_bbox = inner_right_spec.get_position(fig)

    pos_top = ax_top.get_position()
    pos_bot = ax_bot.get_position()

    renderer = fig.canvas.get_renderer()
    bot_tight = ax_bot.get_tightbbox(renderer).transformed(fig.transFigure.inverted())

    x0 = right_bbox.x0
    width = right_bbox.width
    y_bottom = bot_tight.y0
    y_top = pos_top.y1
    full_h = y_top - y_bottom
    gap = 0.01

    # ---- Load pooled held-out geometry (same 40k positions in both panels) ----
    beliefs_true = gpool[f'{hmm}__{param}_true']
    beliefs_pred = gpool[f'{hmm}__{param}_pred']
    n_states = beliefs_true.shape[1]

    if n_states <= 3:
        v = np.array([[0, 0], [1, 0], [0.5, np.sqrt(3)/2]])
        project = lambda b: b @ v
    else:
        pca = PCA(n_components=2).fit(gd[f'{hmm}__{param}_true'])   # same frame as plots/r2.pdf
        project = pca.transform

    half_h = (full_h - gap) / 2

    # ---- panel box sized from the EMPIRICAL points; ground truth gets the identical box ----
    emp_xy = project(beliefs_pred)
    data_w = np.ptp(emp_xy[:, 0])
    data_h = np.ptp(emp_xy[:, 1])
    data_aspect = data_h / data_w if data_w > 0 else 1.0

    fig_w, fig_h = fig.get_size_inches()

    avail_w_in = width * fig_w
    avail_h_in = half_h * fig_h

    if avail_h_in / avail_w_in > data_aspect:
        panel_w = width
        panel_h = width * data_aspect * (fig_w / fig_h)
    else:
        panel_h = half_h
        panel_w = half_h / data_aspect * (fig_h / fig_w)

    x_offset = (width - panel_w) / 2
    y_offset_top = (half_h - panel_h) / 2

    ax_emp = fig.add_axes([x0 + x_offset, y_bottom + half_h + gap + y_offset_top, panel_w, panel_h])
    ax_gt = fig.add_axes([x0 + x_offset, y_bottom + y_offset_top, panel_w, panel_h])

    sub = r2_df[(r2_df['hmm'] == hmm) & (r2_df['param'] == param)]
    best_layer = int(sub[sub['target'] == 'real'].groupby('layer')['R2'].mean().idxmax())

    zorders = {'shuffle': 1, 'real': 2, 'random': 3}   # red (random) on top of blue (shuffle)
    for target, (color, lw, label, ls) in target_styles.items():
        s = sub[sub['target'] == target]
        stats = s.groupby('layer')['R2'].agg(['mean', 'sem']).reset_index()
        for ax in [ax_top, ax_bot]:
            ax.plot(stats['layer'], stats['mean'], color=color, lw=lw, ls=ls,
                    zorder=zorders[target])
            ax.fill_between(stats['layer'],
                            stats['mean'] - 1.96 * stats['sem'],
                            stats['mean'] + 1.96 * stats['sem'],
                            color=color, alpha=0.15, linewidth=0,
                            zorder=zorders[target] - 0.5)

    real_vals = sub[sub['target'] == 'real'].groupby('layer')['R2'].mean()
    ctrl_vals = sub[sub['target'].isin(['shuffle', 'random'])].groupby('layer')['R2'].mean()

    ax_top.set_ylim(real_vals.min() - 0.02, real_vals.max() + 0.01)
    ctrl_range = ctrl_vals.max() - ctrl_vals.min()
    ax_bot.set_ylim(ctrl_vals.min() - 0.15 * ctrl_range, max(ctrl_vals.max() + 0.15 * ctrl_range, 0.15))

    ax_top.yaxis.set_major_locator(plt.MaxNLocator(4))
    ax_bot.yaxis.set_major_locator(plt.MaxNLocator(3))

    n_layers = int(sub['layer'].max())
    xticks = np.arange(0, n_layers + 1, 5)
    for ax in [ax_top, ax_bot]:
        ax.grid(True, alpha=0.2, linewidth=0.5)
        ax.patch.set_edgecolor('none')
        ax.set_xticks(xticks)
        ax.set_xlim(-0.5, n_layers + 0.5)

    ax_top.spines['bottom'].set_visible(False)
    ax_bot.spines['top'].set_visible(False)
    ax_top.tick_params(axis='x', bottom=False, labelbottom=False)
    ax_top.tick_params(axis='y', right=False, labelright=False)
    ax_bot.tick_params(axis='y', right=False, labelright=False)

    spine_lw = matplotlib.rcParams['axes.linewidth']
    kwargs = dict(color='k', clip_on=False, lw=spine_lw)
    for x_frac in [0, 1]:
        ax_top.plot((x_frac - 0.015, x_frac + 0.015), (-0.015, -0.015),
                    transform=ax_top.transAxes, **kwargs)
        ax_bot.plot((x_frac - 0.015, x_frac + 0.015), (1.015, 1.015),
                    transform=ax_bot.transAxes, **kwargs)

    best_r2 = real_vals[best_layer]
    ax_top.plot(best_layer, best_r2, 'o', color=CONN_COLOR, markeredgecolor=CONN_COLOR,
                markersize=5, markeredgewidth=0, zorder=10)

    ax_top.set_title(hmm, fontsize=16)
    ax_top.tick_params(axis='y', labelsize=13, length=3)
    ax_bot.tick_params(axis='both', labelsize=13, length=3)

    if row == 1:
        ax_bot.set_xlabel('Layer', fontsize=14)

    if col == 0:
        mid_y = (pos_top.y1 + pos_bot.y0) / 2
        fig.text(pos_bot.x0 - 0.05, mid_y, 'R²', fontsize=15,
                 va='center', ha='center', rotation=90)

    # ---- Geometry panels: identical boxes, each zoomed to its own points, opacity by density ----
    for ax, beliefs, title in [(ax_emp, beliefs_pred, 'Empirical'),
                                (ax_gt, beliefs_true, 'Ground-Truth')]:
        xy = project(beliefs)
        alpha = density_alpha(xy)
        order = np.argsort(alpha)                                    # sparse first, dense on top
        rgba = np.c_[np.clip(beliefs[:, :3], 0, 1), alpha][order]
        ax.scatter(xy[order, 0], xy[order, 1], s=GEOM_PT, c=rgba, rasterized=True)

        pad = 0.05 * max(np.ptp(xy[:, 0]), np.ptp(xy[:, 1]))
        ax.set_xlim(xy[:, 0].min() - pad, xy[:, 0].max() + pad)
        ax.set_ylim(xy[:, 1].min() - pad, xy[:, 1].max() + pad)
        ax.set_aspect('equal', adjustable='datalim')   # box stays fixed; limits stretch to fill it
        ax.set_xticks([])
        ax.set_yticks([])
        ax.tick_params(left=False, right=False, top=False, bottom=False,
                       labelleft=False, labelright=False, labeltop=False, labelbottom=False)
        ax.xaxis.set_visible(False)
        ax.yaxis.set_visible(False)

        ls = '--' if title == 'Empirical' else '-'
        for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_edgecolor(CONN_COLOR)
            spine.set_linewidth(1.2)
            spine.set_linestyle(ls)

        ax.annotate(title, xy=(1.05, 0.5), xycoords='axes fraction',
                    fontsize=13, rotation=270, va='center', ha='left')

    ax_emp.annotate(f'R² = {best_r2:.2f}', xy=(0.5, 1.05), xycoords='axes fraction',
                    fontsize=13, color='black', ha='center', va='bottom')

    from matplotlib.patches import ConnectionPatch
    for y_frac in [0, 1]:
        con = ConnectionPatch(
            xyA=(best_layer, best_r2), coordsA=ax_top.transData,
            xyB=(0, y_frac), coordsB=ax_emp.transAxes,
            color=CONN_COLOR, ls='--', lw=1.2)
        fig.add_artist(con)

plt.savefig(f'{PLOT_DIR}/r2_pooled.pdf', bbox_inches='tight')
plt.show()
plt.close()

## Figure 1 (b), middle panel

In [ ]:
# ══ Figure 1 panel: Wing R² plot with geometry insets (short, narrow, legend below) ══
#    insets = held-out predictions pooled over the 10 sequences (4000 positions each), as in Figure 3
from matplotlib.lines import Line2D
from matplotlib.patches import ConnectionPatch
import os

HMM_F1 = 'Wing'
PARAM_F1 = [p for h, p in TARGETS if h == HMM_F1][0]

gpool = np.load(os.path.join(RESULTS_DIR, 'geompool_all_qwen35_9b.npz'), allow_pickle=True)
r2_df = pd.read_csv(os.path.join(RESULTS_DIR, 'r2_qwen35_9b.csv'))

target_styles = {
    'real':    ('black',   6.0, 'Activations → Beliefs',          '-'),
    'shuffle': (tab10[0],  6.0, 'Activations → Shuffled Beliefs', '-'),
    'random':  (tab10[3],  6.0, 'Activations → Random Beliefs',   ':'),
}
CONN_COLOR = '0.55'
PANEL_LW = 5.0
INSET_LW = 5.5
GEOM_PT, GEOM_ALPHA = 0.3, 0.3      # 40k points per inset

fig = plt.figure(figsize=(8.5, 3.7))
fig.patch.set_facecolor('white')
outer = fig.add_gridspec(1, 1, top=0.97)

legend_handles = [Line2D([], [], color=c, lw=8, ls=ls, label=lab)
                  for t, (c, lw, lab, ls) in target_styles.items()]
fig.legend(handles=legend_handles, loc='lower center', ncol=1,
           fontsize=30, frameon=False, bbox_to_anchor=(0.5, -0.78),
           handletextpad=0.5, handlelength=1.4, labelspacing=0.35)

inner = outer[0].subgridspec(1, 2, width_ratios=[1.8, 1.2], wspace=0.05)
inner_left = inner[0].subgridspec(2, 1, height_ratios=[70, 30], hspace=0.08)
ax_top = fig.add_subplot(inner_left[0])
ax_bot = fig.add_subplot(inner_left[1])
inner_right_spec = inner[1]

fig.canvas.draw()
renderer = fig.canvas.get_renderer()
right_bbox = inner_right_spec.get_position(fig)
pos_top = ax_top.get_position()
pos_bot = ax_bot.get_position()
bot_tight = ax_bot.get_tightbbox(renderer).transformed(fig.transFigure.inverted())

x0 = right_bbox.x0
width = right_bbox.width
y_bottom = bot_tight.y0
y_top = pos_top.y1
full_h = y_top - y_bottom
gap = 0.02

beliefs_true = gpool[f'{HMM_F1}__{PARAM_F1}_true']
beliefs_pred = gpool[f'{HMM_F1}__{PARAM_F1}_pred']
n_states = beliefs_true.shape[1]
v = np.array([[0, 0], [1, 0], [0.5, np.sqrt(3)/2]])

# panel box sized from the empirical points; ground truth gets the identical box
half_h = (full_h - gap) / 2
emp_xy = beliefs_pred @ v
data_aspect = np.ptp(emp_xy[:, 1]) / np.ptp(emp_xy[:, 0])
fig_w, fig_h = fig.get_size_inches()
if (half_h * fig_h) / (width * fig_w) > data_aspect:
    panel_w = width
    panel_h = width * data_aspect * (fig_w / fig_h)
else:
    panel_h = half_h
    panel_w = half_h / data_aspect * (fig_h / fig_w)
x_offset = (width - panel_w) / 2
y_offset = (half_h - panel_h) / 2

ax_emp = fig.add_axes([x0 + x_offset, y_bottom + half_h + gap + y_offset, panel_w, panel_h])
ax_gt  = fig.add_axes([x0 + x_offset, y_bottom + y_offset - 0.35 * panel_h, panel_w, panel_h])

sub = r2_df[(r2_df['hmm'] == HMM_F1) & (r2_df['param'] == PARAM_F1)]
best_layer = int(sub[sub['target'] == 'real'].groupby('layer')['R2'].mean().idxmax())

zorders = {'shuffle': 1, 'real': 2, 'random': 3}
for target, (color, lw, label, ls) in target_styles.items():
    s = sub[sub['target'] == target]
    stats = s.groupby('layer')['R2'].agg(['mean', 'sem']).reset_index()
    for ax in [ax_top, ax_bot]:
        ax.plot(stats['layer'], stats['mean'], color=color, lw=lw, ls=ls,
                zorder=zorders[target])
        ax.fill_between(stats['layer'],
                        stats['mean'] - 1.96 * stats['sem'],
                        stats['mean'] + 1.96 * stats['sem'],
                        color=color, alpha=0.15, linewidth=0,
                        zorder=zorders[target] - 0.5)

real_vals = sub[sub['target'] == 'real'].groupby('layer')['R2'].mean()
ctrl_vals = sub[sub['target'].isin(['shuffle', 'random'])].groupby('layer')['R2'].mean()
ax_top.set_ylim(real_vals.min() - 0.02, real_vals.max() + 0.01)
ctrl_range = ctrl_vals.max() - ctrl_vals.min()
ax_bot.set_ylim(ctrl_vals.min() - 0.15 * ctrl_range, max(ctrl_vals.max() + 0.15 * ctrl_range, 0.15))
ax_top.yaxis.set_major_locator(plt.MaxNLocator(4))
ax_bot.yaxis.set_major_locator(plt.MaxNLocator(2))

n_layers = int(sub['layer'].max())
for ax in [ax_top, ax_bot]:
    ax.grid(True, alpha=0.2, linewidth=0.5)
    ax.set_xticks(np.arange(0, n_layers + 1, 10))
    ax.xaxis.set_minor_locator(plt.MultipleLocator(5))
    ax.set_xlim(-0.5, n_layers + 0.5)
    for spine in ax.spines.values():
        spine.set_linewidth(PANEL_LW)
ax_top.spines['bottom'].set_visible(False)
ax_bot.spines['top'].set_visible(False)
ax_top.tick_params(axis='x', bottom=False, labelbottom=False, which='both')

kwargs = dict(color='k', clip_on=False, lw=PANEL_LW)
for x_frac in [0, 1]:
    ax_top.plot((x_frac - 0.015, x_frac + 0.015), (-0.015, -0.015),
                transform=ax_top.transAxes, **kwargs)
    ax_bot.plot((x_frac - 0.015, x_frac + 0.015), (1.015, 1.015),
                transform=ax_bot.transAxes, **kwargs)

best_r2 = real_vals[best_layer]
ax_top.plot(best_layer, best_r2, 'o', color=CONN_COLOR, markersize=9,
            markeredgewidth=0, zorder=10)
ax_top.tick_params(axis='y', labelsize=30, length=7, width=2.5)
ax_bot.tick_params(axis='both', labelsize=30, length=7, width=2.5)
ax_bot.set_xlabel('Layer', fontsize=34)
mid_y = (pos_top.y1 + pos_bot.y0) / 2
fig.text(pos_bot.x0 - 0.175, mid_y, 'R²', fontsize=36, va='center', ha='center', rotation=90)

# geometry insets: identical boxes, each zoomed to its own points
for ax, beliefs, title in [(ax_emp, beliefs_pred, 'Empirical'),
                            (ax_gt, beliefs_true, 'Ground-Truth')]:
    xy = beliefs @ v
    ax.scatter(xy[:, 0], xy[:, 1], s=GEOM_PT, c=np.clip(beliefs[:, :3], 0, 1),
               alpha=GEOM_ALPHA, rasterized=True)
    pad = 0.05 * max(np.ptp(xy[:, 0]), np.ptp(xy[:, 1]))
    ax.set_xlim(xy[:, 0].min() - pad, xy[:, 0].max() + pad)
    ax.set_ylim(xy[:, 1].min() - pad, xy[:, 1].max() + pad)
    ax.set_aspect('equal', adjustable='datalim')
    ax.xaxis.set_visible(False); ax.yaxis.set_visible(False)
    ls = '--' if title == 'Empirical' else '-'
    for spine in ax.spines.values():
        spine.set_visible(True); spine.set_edgecolor(CONN_COLOR)
        spine.set_linewidth(INSET_LW); spine.set_linestyle(ls)
    ax.annotate(title, xy=(1.07, 0.5), xycoords='axes fraction',
                fontsize=28, rotation=270, va='center', ha='left')
# ax_emp.annotate(f'R² = {best_r2:.2f}', xy=(0.5, 1.08), xycoords='axes fraction',
#                 fontsize=28, color='black', ha='center', va='bottom')

for y_frac in [0, 1]:
    fig.add_artist(ConnectionPatch(
        xyA=(best_layer, best_r2), coordsA=ax_top.transData,
        xyB=(0, y_frac), coordsB=ax_emp.transAxes,
        color=CONN_COLOR, ls='--', lw=2.6))

plt.savefig(f'{PLOT_DIR}/fig1_r2_wing.svg', bbox_inches='tight', dpi=300)
plt.show(); plt.close()

## Appendix D: geometry grids for all 40 HMMs (Figures 11 to 15)

In [ ]:
# ══ Pooled geometry grids (Figures 11 to 15): late / in-sample late / early / shuffled / random, Qwen3.5-9B ══
#   real             : Section-5 probe per sequence, late window held-out (4000/seq), 10 sequences pooled
#                      -> geompool_all_{MODEL}.npz; label = the R² curve's seed-mean at the best layer
#   insample         : probe fit in fp64 on ALL 5000 late positions, drawn on those same positions, pooled over the
#                      10 sequences -> geompool_insample_{MODEL}.npz. No R² label.
#   early_5k_first250: probe fit on 1000 of the first 5000 positions, its held-out positions
#                      in [EARLY_MIN, 250) drawn (~200/seq)                                    -> early_context/
#                      early labels = seed-mean R² of exactly the drawn predictions
#   shuffle / random : control probes, late window held-out, pooled                            -> geompool_ctl_{MODEL}.npz
#                      label = r2_{MODEL}.csv seed-mean for that target at the best layer
#   3-state families: simplex projection. Arch: PCA fit on the seed-0 late-window true beliefs (paper frame).
#   Every panel's axes contain ALL of its plotted points (data limits + 5%), nothing clipped.
#   Point colour = belief (RGB); point opacity = local point density (2-D histogram, lightly smoothed),
#   so dense structure is drawn solid and sparse off-target predictions fade. Dense points are drawn last.
import os, glob, numpy as np, pandas as pd
from sklearn.decomposition import PCA
from scipy.ndimage import gaussian_filter
MODEL = 'qwen35_9b'
EARLY_MIN = 0                       # first drawn position for the early conditions (set 16 to drop the undetermined belief states)
PT_BIG, PT_SMALL = 0.08, 0.5        # ~40k points per panel vs a few thousand
NROW, NCOL_FAM = 5, 2
v3 = np.array([[0, 0], [1, 0], [0.5, np.sqrt(3) / 2]])

# density -> opacity
DENS_SMOOTH = 1.0            # gaussian smoothing of the histogram, in bins
DENS_GAMMA = 0.5             # opacity = (density / max density) ** gamma; smaller gamma flattens the contrast
ALPHA_MIN, ALPHA_MAX = 0.03, 0.9

def density_alpha(xy):
    """per-point opacity from the smoothed 2-D histogram density at each point"""
    bins = 150 if len(xy) > 10000 else 60
    H, xe, ye = np.histogram2d(xy[:, 0], xy[:, 1], bins=bins)
    H = gaussian_filter(H, DENS_SMOOTH)
    ix = np.clip(np.searchsorted(xe, xy[:, 0], side='right') - 1, 0, bins - 1)
    iy = np.clip(np.searchsorted(ye, xy[:, 1], side='right') - 1, 0, bins - 1)
    d = H[ix, iy]; d = d / d.max()
    return ALPHA_MIN + (ALPHA_MAX - ALPHA_MIN) * d ** DENS_GAMMA

gd = np.load(os.path.join(RESULTS_DIR, f'geom_{MODEL}.npz'), allow_pickle=True)      # sequence-0 late-window beliefs (PCA frame) and each parametrization's best layer
gp_all = np.load(os.path.join(RESULTS_DIR, f'geompool_all_{MODEL}.npz'), allow_pickle=True)
gp_ctl = np.load(os.path.join(RESULTS_DIR, f'geompool_ctl_{MODEL}.npz'), allow_pickle=True)
_ins_path = os.path.join(RESULTS_DIR, f'geompool_insample_{MODEL}.npz')
gp_ins = np.load(_ins_path, allow_pickle=True)
r2df = pd.read_csv(os.path.join(RESULTS_DIR, f'r2_{MODEL}.csv'))

def _r2(Y, P):
    Y = Y.astype(np.float64); P = P.astype(np.float64)
    return 1 - ((Y - P) ** 2).sum() / ((Y - Y.mean(0)) ** 2).sum()
def _csv_r2(hmm, pp, L, target):
    s = r2df[(r2df.hmm == hmm) & (r2df.param == pp) & (r2df.layer == L) & (r2df.target == target)]
    return float(s.R2.mean())

fam_params = {h: sorted({k[len(h)+2:-len('_best_layer')] for k in gd.files
                         if k.startswith(h + '__') and k.endswith('_best_layer')}) for h in HMM_ORDER}

# ── early-context versions: (name, directory, mode, max position drawn, title) ──
EARLY_CONDS = [
    ('early_5k_first250', os.path.join(RESULTS_DIR, 'early_context'), 'window',   250,  'Belief state probe (first 250 tokens, probe fit on the first 5,000)'),
]
early = {}
for name, d, mode, tmax, _ in EARLY_CONDS:
    early[name] = {}
    for f in glob.glob(os.path.join(d, f'early_context_{MODEL}__*.npz')):
        _, hmm, pp = os.path.basename(f)[:-4].split('__'); z = np.load(f)
        true = z['true_early_real']; S = true.shape[0]
        if mode == 'transfer':
            pred = z['pred_early_real'][:, 0].astype(np.float32)                 # late probe verbatim, all early positions
            P = [pred[s, EARLY_MIN:tmax] for s in range(S)]; Y = [true[s, EARLY_MIN:tmax] for s in range(S)]
        else:
            W = int(z['windows'][0][0]); idx = z[f'w{W}_idx_te']
            pred = z[f'w{W}_pred_real'][:, 0].astype(np.float32)                 # held-out predictions of the window probe
            keep = [(idx[s] >= EARLY_MIN) & (idx[s] < tmax) for s in range(S)]
            P = [pred[s][keep[s]] for s in range(S)]; Y = [true[s][idx[s][keep[s]]] for s in range(S)]
        early[name][(hmm, pp)] = (np.concatenate(P), np.concatenate(Y), float(np.mean([_r2(y, p) for y, p in zip(Y, P)])))
    assert len(early[name]) == 40, f'{name}: {len(early[name])} files found in {d}'

def load(cond, hmm, pp, bl):
    """-> pred (n, k), true (n, k), label R² (None = no label)"""
    if cond == 'real':
        return gp_all[f'{hmm}__{pp}_pred'], gp_all[f'{hmm}__{pp}_true'], _csv_r2(hmm, pp, bl, 'real')
    if cond == 'insample':
        return gp_ins[f'{hmm}__{pp}_pred'], gp_ins[f'{hmm}__{pp}_true'], None
    if cond in early:
        return early[cond][(hmm, pp)]
    return gp_ctl[f'{hmm}__{pp}_{cond}_pred'], gp_ctl[f'{hmm}__{pp}_{cond}_true'], _csv_r2(hmm, pp, bl, cond)

def _wrap(pp):
    t = pp.split(', ')
    return t[0] if len(t) == 1 else t[0] + '\n' + ', '.join(t[1:])

CONDS = [('real',     'Belief state probe'),
         ('insample', 'Belief state probe (fit on all 5,000 late positions, in-sample)')] + \
        [(n, t) for n, _, _, _, t in EARLY_CONDS] + \
        [('shuffle',  'Shuffled belief state probe'),
         ('random',   'Random belief state probe')]

for cond, ttl in CONDS:
    fig = plt.figure(figsize=(1.7 * NCOL_FAM * len(HMM_ORDER), 2.2 * NROW))
    fig.suptitle(ttl, fontsize=20, y=1.03)
    subfigs = fig.subfigures(1, len(HMM_ORDER), wspace=0.04)
    for sf, hmm in zip(subfigs, HMM_ORDER):
        params = fam_params[hmm]
        axes = sf.subplots(NROW, NCOL_FAM, squeeze=False,
                           gridspec_kw=dict(hspace=0.65, wspace=0.15, top=0.92, bottom=0.05))
        for p_idx in range(NROW * NCOL_FAM):
            r, c = p_idx % NROW, p_idx // NROW
            ax = axes[r, c]; ax.set_xticks([]); ax.set_yticks([])
            for sp in ax.spines.values(): sp.set_visible(False)
            if p_idx >= len(params): ax.axis('off'); continue
            pp = params[p_idx]; bl = int(gd[f'{hmm}__{pp}_best_layer'])
            pred, true, r2 = load(cond, hmm, pp, bl)
            if pred.shape[1] == 3:
                project = lambda b: b @ v3
            else:
                project = PCA(n_components=2).fit(gd[f'{hmm}__{pp}_true']).transform
            xy = project(pred)
            alpha = density_alpha(xy)
            order = np.argsort(alpha)                                        # sparse first, dense on top
            rgba = np.c_[np.clip(pred[:, :3], 0, 1), alpha][order]
            ax.scatter(xy[order, 0], xy[order, 1], s=PT_BIG if len(xy) > 10000 else PT_SMALL,
                       c=rgba, rasterized=True)
            pad = 0.05 * max(np.ptp(xy[:, 0]), np.ptp(xy[:, 1]))          # all points fit, nothing clipped
            ax.set_xlim(xy[:, 0].min() - pad, xy[:, 0].max() + pad)
            ax.set_ylim(xy[:, 1].min() - pad, xy[:, 1].max() + pad)
            ax.set_aspect('equal', adjustable='box')
            ax.set_title(_wrap(pp), fontsize=9, pad=6)
            ax.set_xlabel(f'(L{bl})' if r2 is None else f'$R^2$={r2:.2f} (L{bl})',
                          fontsize=9, color='0.2', labelpad=2)
        sf.suptitle(hmm, fontsize=20, y=0.99)
    fig.savefig(f'{PLOT_DIR}/geometry_grid_{cond}_pooled_{MODEL}.pdf', bbox_inches='tight')
    plt.show(); plt.close()

## Appendix I: probe R² with controls for all models (Figures 29 to 34)

In [ ]:
for model_key, df in r2_files.items():
    model_name = MODEL_LABELS.get(model_key, model_key)
    data = df[df['hmm'] != 'Spiral']
    hmms = [h for h in HMM_ORDER if h in data['hmm'].values]

    # Shared y-ranges
    real_global_min = data[data['target'] == 'real'].groupby(['hmm', 'param', 'layer'])['R2'].mean().min()
    ctrl = data[data['target'].isin(['shuffle', 'random'])]
    ctrl_means = ctrl.groupby(['hmm', 'param', 'layer'])['R2'].mean()
    ctrl_lo, ctrl_hi = ctrl_means.min(), ctrl_means.max()
    ctrl_pad = 0.15 * max(ctrl_hi - ctrl_lo, 0.01)

    fig = plt.figure(figsize=(13, 8))
    fig.patch.set_facecolor('white')
    fig.patch.set_edgecolor('white')
    fig.patch.set_linewidth(0)
    outer = fig.add_gridspec(2, 2, wspace=0.55, hspace=0.3)

    axes_pairs = []
    for j, hmm_name in enumerate(hmms[:4]):
        r, c = divmod(j, 2)
        inner = outer[r, c].subgridspec(2, 1, height_ratios=[70, 30], hspace=0.08)
        ax_top = fig.add_subplot(inner[0])
        ax_bot = fig.add_subplot(inner[1])
        axes_pairs.append((ax_top, ax_bot, hmm_name, r, c))

        hmm_data = data[data['hmm'] == hmm_name]
        params = sorted(hmm_data['param'].unique())
        n_p = len(params)
        tp = np.linspace(0.3, 0.9, n_p)
        colors_real = plt.cm.viridis(tp)
        colors_shuf = plt.cm.Blues(tp)
        colors_rand = plt.cm.Reds(tp)

        for i, param in enumerate(params):
            for target, colors, ls, lw, alpha in [
                ('shuffle', colors_shuf, '-', 0.6, 0.5),
                ('random',  colors_rand, '-', 0.6, 0.5),
                ('real',    colors_real, '-', 1.5, 1.0),
            ]:
                sub = hmm_data[(hmm_data['param'] == param) & (hmm_data['target'] == target)]
                stats = sub.groupby('layer')['R2'].agg(['mean', 'sem']).reset_index()
                for ax in [ax_top, ax_bot]:
                    ax.plot(stats['layer'], stats['mean'], ls=ls, color=colors[i],
                            lw=lw, alpha=alpha)
                    ax.fill_between(stats['layer'],
                                    stats['mean'] - 1.96 * stats['sem'],
                                    stats['mean'] + 1.96 * stats['sem'],
                                    color=colors[i], alpha=0.15, linewidth=0)

        # Track actual plotted bounds
        real_lo, real_hi = np.inf, -np.inf
        ctrl_lo_panel, ctrl_hi_panel = np.inf, -np.inf
        for i, param in enumerate(params):
            for target in ['real', 'shuffle', 'random']:
                sub = hmm_data[(hmm_data['param'] == param) & (hmm_data['target'] == target)]
                stats = sub.groupby('layer')['R2'].agg(['mean', 'sem']).reset_index()
                lo = (stats['mean'] - 1.96 * stats['sem']).min()
                hi = (stats['mean'] + 1.96 * stats['sem']).max()
                if target == 'real':
                    real_lo = min(real_lo, lo)
                    real_hi = max(real_hi, hi)
                else:
                    ctrl_lo_panel = min(ctrl_lo_panel, lo)
                    ctrl_hi_panel = max(ctrl_hi_panel, hi)

        real_pad = 0.02 * (real_hi - real_lo)
        ctrl_pad = 0.05 * (ctrl_hi_panel - ctrl_lo_panel)
        ax_bot.set_ylim(ctrl_lo_panel - ctrl_pad, ctrl_hi_panel + ctrl_pad)
        ax_top.set_ylim(real_lo - real_pad, real_hi + 3 * real_pad)

        ax_top.spines['bottom'].set_visible(False)
        ax_bot.spines['top'].set_visible(False)
        ax_top.tick_params(axis='x', bottom=False, labelbottom=False)
        ax_top.tick_params(axis='y', labelsize=13, length=3)
        ax_bot.tick_params(axis='both', labelsize=13, length=3)

        n_layers = int(hmm_data['layer'].max())
        xticks = np.arange(0, n_layers + 1, 5)
        for ax in [ax_top, ax_bot]:
            ax.set_xlim(-0.5, n_layers + 0.5)
            ax.set_xticks(xticks)
            ax.grid(True, alpha=0.2, linewidth=0.5)
            ax.patch.set_edgecolor('none')

        d = 0.015
        bk = dict(color='k', clip_on=False, lw=matplotlib.rcParams['axes.linewidth'])
        for x_frac in [0, 1]:
            ax_top.plot((x_frac - d, x_frac + d), (-d, -d),
                        transform=ax_top.transAxes, **bk)
            ax_bot.plot((x_frac - d, x_frac + d), (1 + d, 1 + d),
                        transform=ax_bot.transAxes, **bk)

        ax_top.set_title(hmm_name, fontsize=16)
        if r == 1: ax_bot.set_xlabel('Layer', fontsize=14)

        # Legend to the right — sits in the wspace gap
        param_handles = [Line2D([], [], color=colors_real[i], lw=3, label=p)
                         for i, p in enumerate(params)]
        param_handles.append(Line2D([], [], color='tab:blue', lw=3, label='Shuffled'))
        param_handles.append(Line2D([], [], color='tab:red', lw=3, label='Random'))
        ax_top.legend(handles=param_handles, fontsize=9.5, ncol=1,
                      loc='center left', bbox_to_anchor=(1.02, 0.3),
                      frameon=True, edgecolor='0.8',
                      borderpad=0.4, handlelength=1.2, labelspacing=0.3)

    # R² ylabel centered across each row
    fig.canvas.draw()
    for row_idx in range(2):
        row_axes = [(at, ab) for at, ab, _, r, c in axes_pairs if r == row_idx and c == 0]
        if not row_axes: continue
        at, ab = row_axes[0]
        y_mid = (at.get_position().y1 + ab.get_position().y0) / 2
        x_left = ab.get_position().x0
        fig.text(x_left - 0.05, y_mid, 'R²', fontsize=15,
                 va='center', ha='center', rotation=90)

    fig.suptitle(model_name, fontsize=16, y=0.98)
    plt.savefig(f'{PLOT_DIR}/r2_controls_{model_key}.pdf', bbox_inches='tight')
    plt.show(); plt.close()

## Figure 4 and Appendix J: transfer probes (Figures 35 to 40)

In [ ]:
# Load cross-R² data — change model key as needed
MODEL = 'qwen35_9b'
cross_df = harmonize_cross(load_csv('transfer_probes', MODEL))
gt_df = harmonize_cross(load_csv('transfer_probes_gt', MODEL))
cross_df, gt_df = harmonize_wing_labels(cross_df, gt_df)

# Exclude Spiral
if cross_df is not None: cross_df = cross_df[cross_df['hmm'] != 'Spiral']
if gt_df is not None: gt_df = gt_df[gt_df['hmm'] != 'Spiral']

def fmt_tick(v):
    if abs(v - round(v)) < 1e-9:
        return str(int(round(v)))          # 1.0 / 0.99999 -> "1", 10 -> "10"
    s = f'{v:.2f}'.rstrip('0').rstrip('.')  # ".rstrip('.')" kills any leftover dot
    if -1 < v < 1:
        s = s.lstrip('0')                   # 0.9 -> ".9"
    return s

In [ ]:
from matplotlib.ticker import MaxNLocator

cross_df = cross_df[cross_df['hmm'] != 'Spiral']
gt_df = gt_df[gt_df['hmm'] != 'Spiral']

hmm_order = ['Mess3', 'Arch', 'Wing', 'Strata']

VERT_COLOR = 'blue'
diag_color = 'black'
offdiag_color = tab10[3]

# figsize height = 7.6 * (0.75 + 1.8) / (0.75 + 1.8 + 1) so rows 0,1 keep their size
fig = plt.figure(figsize=(16, 6.1))
fig.patch.set_facecolor('white')
fig.patch.set_edgecolor('white')
fig.patch.set_linewidth(0)

outer = fig.add_gridspec(2, 4, height_ratios=[0.75, 1.8], hspace=0.3, wspace=0.3)

# Precompute
all_pearson = []
precomputed = {}

for hmm in hmm_order:
    c = cross_df[cross_df['hmm'] == hmm]
    g = gt_df[gt_df['hmm'] == hmm]
    params = sorted(c['train'].unique())
    n_params = len(params)
    layers = sorted(c['layer'].unique())

    gt_matrix = np.zeros((n_params, n_params))
    for pi, p_train in enumerate(params):
        for pj, p_test in enumerate(params):
            gt_matrix[pi, pj] = g[(g['train'] == p_train) & (g['test'] == p_test)]['R2'].mean()

    gt_vec = gt_matrix.ravel()   # correlate the FULL heatmaps (all cells)

    pearson_vals = []
    emp_matrices = {}
    for layer in layers:
        emp_matrix = np.zeros((n_params, n_params))
        for pi, p_train in enumerate(params):
            for pj, p_test in enumerate(params):
                emp_matrix[pi, pj] = c[(c['train'] == p_train) & (c['test'] == p_test) & (c['layer'] == layer)]['R2'].mean()
        emp_matrices[layer] = emp_matrix
        emp_vec = emp_matrix.ravel()
        pr, _ = pearsonr(gt_vec, emp_vec)
        pearson_vals.append(pr)

    pearson_vals = np.array(pearson_vals)
    all_pearson.extend(pearson_vals)

    best_layer_idx = np.argmax(pearson_vals)
    best_layer = layers[best_layer_idx]

    hmm_vmin = min(gt_matrix.min(), emp_matrices[best_layer].min())
    hmm_vmax = max(gt_matrix.max(), emp_matrices[best_layer].max())

    precomputed[hmm] = {
        'layers': layers, 'params': params,
        'pearson': pearson_vals,
        'gt_matrix': gt_matrix, 'emp_best': emp_matrices[best_layer],
        'best_layer': best_layer,
        'vmin': hmm_vmin, 'vmax': hmm_vmax,
    }

# Shared y limits (Pearson only)
shared_ylim = (min(all_pearson) - 0.05, max(all_pearson) + 0.02)

for i, hmm in enumerate(hmm_order):
    d = precomputed[hmm]
    layers = d['layers']
    best_layer = d['best_layer']

    ax_corr = fig.add_subplot(outer[0, i])

    inner_bot = outer[1, i].subgridspec(1, 2, wspace=0.08)
    ax_gt_hm = fig.add_subplot(inner_bot[0])      # swapped: GT now left
    ax_emp_hm = fig.add_subplot(inner_bot[1])     # empirical now right

    # Correlation curves
    ax_corr.plot(layers, d['pearson'], color='gray', lw=5)
    ax_corr.axvline(best_layer, color=VERT_COLOR, lw=4, zorder=5)

    best_layer_idx = list(layers).index(best_layer)
    best_r = d['pearson'][best_layer_idx]

    # lowered to the axis bottom; Wing & Strata to the right of the line, others to the left
    annot_y = shared_ylim[0] + 0.1
    if hmm in ('Strata', 'Wing'):
        xy = (best_layer + 1, annot_y); ha = 'left'
    else:  # Mess3, Arch -> left of the blue line
        xy = (best_layer - 1, annot_y); ha = 'right'
    ax_corr.annotate(f'r = {best_r:.2f}', xy=xy, fontsize=19,
                     color=VERT_COLOR, ha=ha, va='bottom')

    _lo, _hi = shared_ylim[0], max(shared_ylim[1], 1.0)
    ax_corr.set_ylim(_lo, _hi)
    ax_corr.set_title(hmm, fontsize=19, pad=12)
    ax_corr.tick_params(axis='both', labelsize=19, length=3)
    # evenly spaced ticks anchored at 1.0 (1 minus k*step), 3-4 of them
    _step = MaxNLocator(nbins=3, min_n_ticks=3, steps=[1, 2, 2.5, 5, 10]).tick_values(_lo, 1.0)
    _step = _step[1] - _step[0]
    _ticks = [1.0 - k * _step for k in range(4) if 1.0 - k * _step >= _lo]
    ax_corr.set_yticks(sorted(_ticks))
    ax_corr.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: fmt_tick(v)))
    ax_corr.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: fmt_tick(v)))
    ax_corr.grid(True, alpha=0.2, linewidth=0.5)
    ax_corr.set_xticks([0, 10, 20, 30])
    ax_corr.set_xlabel('Layer', fontsize=19, labelpad=2)
    if i == 0:
        ax_corr.set_ylabel('corr(Empirical,\nGround-Truth)', fontsize=14)

    cmap = plt.cm.RdYlGn

    # Separate ranges for empirical and ground-truth
    emp_vmin, emp_vmax = d['emp_best'].min(), d['emp_best'].max()
    gt_vmin, gt_vmax = d['gt_matrix'].min(), d['gt_matrix'].max()

    im_emp = ax_emp_hm.imshow(d['emp_best'], cmap=cmap, vmin=emp_vmin, vmax=emp_vmax, aspect='equal')
    im_gt = ax_gt_hm.imshow(d['gt_matrix'], cmap=cmap, vmin=gt_vmin, vmax=gt_vmax, aspect='equal')

    for ax in [ax_emp_hm, ax_gt_hm]:
        ax.set_xticks([])
        ax.set_yticks([])
        ax.tick_params(left=False, right=False, top=False, bottom=False)
        ax.xaxis.set_visible(False)
        ax.yaxis.set_visible(False)

        for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_edgecolor(VERT_COLOR)
            spine.set_linewidth(3.5)

    ax_emp_hm.annotate(f'Empirical\n(L{best_layer})', xy=(0.5, 1.08), xycoords='axes fraction',
                        fontsize=16, ha='center', va='bottom', color='blue')
    ax_gt_hm.annotate('Ground-\nTruth', xy=(0.5, 1.08), xycoords='axes fraction',
                        fontsize=16, ha='center', va='bottom', color='blue')

    params = d['params']
    n_params = len(params)

    # Source HMM on left of the LEFT heatmap (now ground-truth)
    ax_gt_hm.yaxis.set_visible(True)
    ax_gt_hm.set_yticks([])
    ax_gt_hm.set_ylabel('Source HMM', fontsize=16)

    # Target HMM centered between the two heatmaps (left = gt, right = emp)
    mid_x = (ax_gt_hm.get_position().x0 + ax_emp_hm.get_position().x1) / 2
    fig.text(mid_x, ax_gt_hm.get_position().y0 - 0.02, 'Target HMM',
             fontsize=16, ha='center', va='top')

    # Two side-by-side colorbars, one under each heatmap
    pos_emp = ax_emp_hm.get_position()
    pos_gt = ax_gt_hm.get_position()
    cbar_h = pos_gt.height / n_params
    cbar_y = pos_gt.y0 - 0.09

    cbar_ax_emp = fig.add_axes([pos_emp.x0, cbar_y, pos_emp.width, cbar_h])
    fig.colorbar(im_emp, cax=cbar_ax_emp, orientation='horizontal')
    cbar_ax_emp.set_xticks([emp_vmin, emp_vmax])
    cbar_ax_emp.set_xticklabels([fmt_tick(emp_vmin), fmt_tick(emp_vmax)])
    cbar_ax_emp.tick_params(labelsize=17, length=2)
    ticks_emp = cbar_ax_emp.get_xticklabels()
    ticks_emp[0].set_ha('left')
    ticks_emp[1].set_ha('right')

    cbar_ax_gt = fig.add_axes([pos_gt.x0, cbar_y, pos_gt.width, cbar_h])
    fig.colorbar(im_gt, cax=cbar_ax_gt, orientation='horizontal')
    cbar_ax_gt.set_xticks([gt_vmin, gt_vmax])
    cbar_ax_gt.set_xticklabels([fmt_tick(gt_vmin), fmt_tick(gt_vmax)])
    cbar_ax_gt.tick_params(labelsize=17, length=2)
    ticks_gt = cbar_ax_gt.get_xticklabels()
    ticks_gt[0].set_ha('left')
    ticks_gt[1].set_ha('right')

    # R² label to the left of the LEFTMOST colorbar (now ground-truth's)
    fig.text(pos_gt.x0 - 0.005, cbar_y + cbar_h / 2, 'R²',
             fontsize=19, ha='right', va='center')

plt.tight_layout()

plt.savefig(f'{PLOT_DIR}/within_r2.pdf', bbox_inches='tight')
plt.show()
plt.close()

In [ ]:
from matplotlib.ticker import MaxNLocator
from matplotlib.offsetbox import TextArea, DrawingArea, HPacker, VPacker, AnchoredOffsetbox
from matplotlib.patches import Rectangle

VERT_COLOR = 'blue'
diag_color = 'black'
offdiag_color = tab10[3]

# row 2 (levels): self = R2(acts_i -> beliefs_i), cross = R2(acts_i -> beliefs_j) for each
# source row's lowest-GT target (the hardest neighbor), averaged across rows.
def _diag_compute(hmm):
    c = cross_df[cross_df['hmm'] == hmm]; g = gt_df[gt_df['hmm'] == hmm]
    prm = sorted(c['train'].unique()); lyrs = sorted(c['layer'].unique())
    gt_pair = {(pt, pe): g[(g['train'] == pt) & (g['test'] == pe)]['R2'].mean()
               for pt in prm for pe in prm if pt != pe}
    row_worst = {pt: min((pe for pe in prm if pe != pt), key=lambda pe: gt_pair[(pt, pe)])
                 for pt in prm}
    worst_pairs = [(prm.index(pt), prm.index(row_worst[pt])) for pt in prm]
    gt_vals = np.concatenate([g[(g['train'] == pt) & (g['test'] == row_worst[pt])]['R2'].values for pt in prm])
    gtm, gtc = gt_vals.mean(), gt_vals.std() / np.sqrt(len(gt_vals))
    sm, ssd, cm, pc = [], [], [], []
    for L in lyrs:
        self_vals = c[(c['train'] == c['test']) & (c['layer'] == L)]['R2'].values
        cross_vals = []
        for pt in prm:
            ovs = c[(c['train'] == pt) & (c['test'] == row_worst[pt]) & (c['layer'] == L)]['R2'].values
            cross_vals.append(ovs)
        cross_vals = np.concatenate(cross_vals)
        sm.append(self_vals.mean());  ssd.append(self_vals.std() / np.sqrt(len(self_vals)))
        cm.append(cross_vals.mean()); pc.append(cross_vals.std() / np.sqrt(len(cross_vals)))
    return (np.array(lyrs), np.array(sm), np.array(ssd), np.array(cm), np.array(pc), gtm, gtc, worst_pairs)

def _draw_diag(ax, data):
    layers, sm, ssd, cm, pc, gtm, gtc, _wp = data
    ax.plot(layers, sm, '-', color=diag_color, lw=5)
    ax.fill_between(layers, sm - 1.96 * ssd, sm + 1.96 * ssd, color=diag_color, alpha=0.15, linewidth=0)
    ax.plot(layers, cm, '-', color=offdiag_color, lw=5)
    ax.fill_between(layers, cm - 1.96 * pc, cm + 1.96 * pc, color=offdiag_color, alpha=0.15, linewidth=0)

for model_key in MODEL_KEYS:
    model_name = MODEL_LABELS.get(model_key, model_key)
    cross_df = harmonize_cross(load_csv('transfer_probes', model_key))
    gt_df = harmonize_cross(load_csv('transfer_probes_gt', model_key))
    if cross_df is None or gt_df is None:
        continue
    cross_df, gt_df = harmonize_wing_labels(cross_df, gt_df)
    cross_df = cross_df[cross_df['hmm'] != 'Spiral']
    gt_df = gt_df[gt_df['hmm'] != 'Spiral']
    hmm_order = [h for h in ['Mess3', 'Arch', 'Wing', 'Strata'] if h in cross_df['hmm'].values]

    fig = plt.figure(figsize=(16, 8.4))
    fig.patch.set_facecolor('white')
    fig.patch.set_edgecolor('white')
    fig.patch.set_linewidth(0)

    outer = fig.add_gridspec(3, 4, height_ratios=[0.75, 1.8, 1], hspace=0.3, wspace=0.3)
    # FIX: lock the layout BEFORE any get_position() call, so the colorbars /
    # fig.texts placed from panel positions inside the loop are not stale.
    fig.subplots_adjust(left=0.06, right=0.98, top=0.92, bottom=0.07)

    # Precompute
    all_pearson = []
    precomputed = {}

    for hmm in hmm_order:
        c = cross_df[cross_df['hmm'] == hmm]
        g = gt_df[gt_df['hmm'] == hmm]
        params = sorted(c['train'].unique())
        n_params = len(params)
        layers = sorted(c['layer'].unique())

        gt_matrix = np.zeros((n_params, n_params))
        for pi, p_train in enumerate(params):
            for pj, p_test in enumerate(params):
                gt_matrix[pi, pj] = g[(g['train'] == p_train) & (g['test'] == p_test)]['R2'].mean()

        gt_vec = gt_matrix.ravel()   # correlate the FULL heatmaps (all cells)

        pearson_vals = []
        emp_matrices = {}
        for layer in layers:
            emp_matrix = np.zeros((n_params, n_params))
            for pi, p_train in enumerate(params):
                for pj, p_test in enumerate(params):
                    emp_matrix[pi, pj] = c[(c['train'] == p_train) & (c['test'] == p_test) & (c['layer'] == layer)]['R2'].mean()
            emp_matrices[layer] = emp_matrix
            emp_vec = emp_matrix.ravel()
            pr, _ = pearsonr(gt_vec, emp_vec)
            pearson_vals.append(pr)

        pearson_vals = np.array(pearson_vals)
        all_pearson.extend(pearson_vals)

        best_layer_idx = np.argmax(pearson_vals)
        best_layer = layers[best_layer_idx]

        hmm_vmin = min(gt_matrix.min(), emp_matrices[best_layer].min())
        hmm_vmax = max(gt_matrix.max(), emp_matrices[best_layer].max())

        precomputed[hmm] = {
            'layers': layers, 'params': params,
            'pearson': pearson_vals,
            'gt_matrix': gt_matrix, 'emp_best': emp_matrices[best_layer],
            'best_layer': best_layer,
            'vmin': hmm_vmin, 'vmax': hmm_vmax,
        }

    # Shared y limits (Pearson only)
    shared_ylim = (min(all_pearson) - 0.05, max(all_pearson) + 0.02)

    diag_axes = {}
    diag_data = {}

    for i, hmm in enumerate(hmm_order):
        d = precomputed[hmm]
        layers = d['layers']
        best_layer = d['best_layer']

        ax_corr = fig.add_subplot(outer[0, i])

        inner_bot = outer[1, i].subgridspec(1, 2, wspace=0.08)
        ax_gt_hm = fig.add_subplot(inner_bot[0])      # CHANGED: GT now left
        ax_emp_hm = fig.add_subplot(inner_bot[1])     # CHANGED: empirical now right

        # Correlation curves
        ax_corr.plot(layers, d['pearson'], color='gray', lw=5)
        ax_corr.axvline(best_layer, color=VERT_COLOR, lw=4, zorder=5)

        best_layer_idx = list(layers).index(best_layer)
        best_r = d['pearson'][best_layer_idx]

        annot_y = shared_ylim[0] + 0.1
        if hmm in ('Strata', 'Wing'):
            xy = (best_layer + 1, annot_y); ha = 'left'
        else:
            xy = (best_layer - 1, annot_y); ha = 'right'
        ax_corr.annotate(f'r = {best_r:.2f}', xy=xy, fontsize=19,
                         color=VERT_COLOR, ha=ha, va='bottom')

        _lo, _hi = shared_ylim[0], max(shared_ylim[1], 1.0)
        ax_corr.set_ylim(_lo, _hi)
        ax_corr.set_title(hmm, fontsize=19, pad=12)
        ax_corr.tick_params(axis='both', labelsize=19, length=3)
        _step = MaxNLocator(nbins=3, min_n_ticks=3, steps=[1, 2, 2.5, 5, 10]).tick_values(_lo, 1.0)
        _step = _step[1] - _step[0]
        _ticks = [1.0 - k * _step for k in range(4) if 1.0 - k * _step >= _lo]
        ax_corr.set_yticks(sorted(_ticks))
        ax_corr.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: fmt_tick(v)))
        ax_corr.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: fmt_tick(v)))
        ax_corr.grid(True, alpha=0.2, linewidth=0.5)
        ax_corr.set_xticks([0, 10, 20, 30])
        ax_corr.set_xlabel('Layer', fontsize=19, labelpad=2)
        if i == 0:
            ax_corr.set_ylabel('corr(Empirical,\nGround-Truth)', fontsize=14)

        cmap = plt.cm.RdYlGn

        emp_vmin, emp_vmax = d['emp_best'].min(), d['emp_best'].max()
        gt_vmin, gt_vmax = d['gt_matrix'].min(), d['gt_matrix'].max()

        im_emp = ax_emp_hm.imshow(d['emp_best'], cmap=cmap, vmin=emp_vmin, vmax=emp_vmax, aspect='equal')
        im_gt = ax_gt_hm.imshow(d['gt_matrix'], cmap=cmap, vmin=gt_vmin, vmax=gt_vmax, aspect='equal')

        _ddata = _diag_compute(hmm); _wp = _ddata[7]
        for _k in range(d['emp_best'].shape[0]):
            ax_emp_hm.add_patch(Rectangle((_k - 0.5, _k - 0.5), 1, 1, fill=False,
                                          edgecolor='black', lw=2, zorder=10))
        for _ri, _cj in _wp:
            ax_emp_hm.add_patch(Rectangle((_cj - 0.5, _ri - 0.5), 1, 1, fill=False,
                                          edgecolor=offdiag_color, lw=2.2, zorder=11))
            ax_gt_hm.add_patch(Rectangle((_cj - 0.5, _ri - 0.5), 1, 1, fill=False,
                                         edgecolor=offdiag_color, lw=2.2, ls='--', zorder=11))

        for ax in [ax_emp_hm, ax_gt_hm]:
            ax.set_xticks([])
            ax.set_yticks([])
            ax.tick_params(left=False, right=False, top=False, bottom=False)
            ax.xaxis.set_visible(False)
            ax.yaxis.set_visible(False)
            for spine in ax.spines.values():
                spine.set_visible(True)
                spine.set_edgecolor(VERT_COLOR)
                spine.set_linewidth(3.5)

        # Labels above heatmaps (two lines; Empirical gets its best layer; centered)
        ax_emp_hm.annotate(f'Empirical\n(L{best_layer})', xy=(0.5, 1.08), xycoords='axes fraction',
                            fontsize=16, ha='center', va='bottom', color='blue')
        ax_gt_hm.annotate('Ground-\nTruth', xy=(0.5, 1.08), xycoords='axes fraction',
                            fontsize=16, ha='center', va='bottom', color='blue')

        params = d['params']
        n_params = len(params)

        # CHANGED: Source HMM on the left heatmap, which is now ground-truth
        ax_gt_hm.yaxis.set_visible(True)
        ax_gt_hm.set_yticks([])
        ax_gt_hm.set_ylabel('Source HMM', fontsize=16)

        # CHANGED: centered between left (gt) and right (emp) heatmaps
        mid_x = (ax_gt_hm.get_position().x0 + ax_emp_hm.get_position().x1) / 2
        fig.text(mid_x, ax_gt_hm.get_position().y0 - 0.02, 'Target HMM',
                 fontsize=16, ha='center', va='top')

        pos_emp = ax_emp_hm.get_position()
        pos_gt = ax_gt_hm.get_position()
        cbar_h = pos_gt.height / n_params
        cbar_y = pos_gt.y0 - 0.075

        cbar_ax_emp = fig.add_axes([pos_emp.x0, cbar_y, pos_emp.width, cbar_h])
        fig.colorbar(im_emp, cax=cbar_ax_emp, orientation='horizontal')
        cbar_ax_emp.set_xticks([emp_vmin, emp_vmax])
        cbar_ax_emp.set_xticklabels([fmt_tick(emp_vmin), fmt_tick(emp_vmax)])
        cbar_ax_emp.tick_params(labelsize=17, length=2)
        ticks_emp = cbar_ax_emp.get_xticklabels()
        ticks_emp[0].set_ha('left')
        ticks_emp[1].set_ha('right')

        cbar_ax_gt = fig.add_axes([pos_gt.x0, cbar_y, pos_gt.width, cbar_h])
        fig.colorbar(im_gt, cax=cbar_ax_gt, orientation='horizontal')
        cbar_ax_gt.set_xticks([gt_vmin, gt_vmax])
        cbar_ax_gt.set_xticklabels([fmt_tick(gt_vmin), fmt_tick(gt_vmax)])
        cbar_ax_gt.tick_params(labelsize=17, length=2)
        ticks_gt = cbar_ax_gt.get_xticklabels()
        ticks_gt[0].set_ha('left')
        ticks_gt[1].set_ha('right')

        # CHANGED: R² label anchors to the leftmost colorbar (now ground-truth's)
        fig.text(pos_gt.x0 - 0.005, cbar_y + cbar_h / 2, 'R²',
                 fontsize=19, ha='right', va='center')

        # ---- row 2: self / cross levels (no title) ----
        data = _diag_compute(hmm)
        ax_diag = fig.add_subplot(outer[2, i])
        _draw_diag(ax_diag, data)
        ax_diag.set_xticks([0, 10, 20, 30]); ax_diag.tick_params(axis='both', labelsize=19, length=3)
        ax_diag.grid(True, alpha=0.2, linewidth=0.5)
        ax_diag.yaxis.set_major_locator(MaxNLocator(nbins=3, min_n_ticks=3))
        ax_diag.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: fmt_tick(v)))
        ax_diag.set_xlabel('Layer', fontsize=19, labelpad=2)
        if i == 0:
            ax_diag.set_ylabel('R²', fontsize=19)
        diag_axes[hmm] = ax_diag; diag_data[hmm] = data

    # ===== nudge ONLY the diag row: pull it UP toward the heatmap row (closes the
    # row2-row3 gap), and make it slightly shorter =====
    RAISE = 0.025
    DIAG_SHRINK = 0.92
    for _hmm in hmm_order:
        _ax = diag_axes[_hmm]; _p = _ax.get_position()
        _new_h = _p.height * DIAG_SHRINK
        _new_top = _p.y1 + RAISE
        _ax.set_position([_p.x0, _new_top - _new_h, _p.width, _new_h])

    # ===== legend (solid black = self, solid red = cross to hardest neighbor) =====
    def _leg_entry(color, ls, prefix, red_suffix, fs=18):
        da = DrawingArea(46, 18, 0, 0)
        da.add_artist(Line2D([2, 44], [9, 9], color=color, lw=5, ls=ls))
        kids = [da, TextArea(prefix, textprops=dict(color='black', fontsize=fs))]
        if red_suffix:
            kids.append(TextArea(red_suffix, textprops=dict(color=offdiag_color, fontsize=fs)))
        return HPacker(children=kids, align='center', pad=0, sep=4)

    entries = [
        _leg_entry(diag_color, '-', r"Activations (HMM $i$) → Beliefs (HMM $i$)", ""),
        _leg_entry(offdiag_color, '-', r"Activations (HMM $i$) → Beliefs ", r"(HMM $j$)"),
    ]
    leg_grid = HPacker(children=[entries[0], entries[1]], align='center', pad=0, sep=40)
    fig.add_artist(AnchoredOffsetbox(loc='upper center', child=leg_grid, frameon=False,
                                     bbox_to_anchor=(0.5, 0.0), bbox_transform=fig.transFigure,
                                     borderpad=0))

    fig.suptitle(model_name, fontsize=22, y=1.0)
    plt.savefig(f'{PLOT_DIR}/within_r2_{model_key}.pdf', bbox_inches='tight')
    plt.show()
    plt.close()

## Figure 5 (top) and Appendix K: k-suffix probes (Figures 41 to 46)

In [ ]:
# Load reduced R² CSV — change model key as needed
MODEL = 'qwen35_9b'
redr2_df = load_csv('ksuffix_probes', MODEL)
print(f'RedR²: {len(redr2_df)} rows') if redr2_df is not None else None


In [ ]:
import matplotlib
from matplotlib.ticker import MaxNLocator
matplotlib.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['DejaVu Sans'],
    'font.weight': 'light',
})

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D

sns.set_context('notebook')

tab10 = sns.color_palette('tab10')

def fmt_tick(v):
    if v == int(v):
        s = f'{v:.1f}'
    else:
        s = f'{v:.2f}'
    if '.' in s and -1 < v < 1:
        s = s.lstrip('0')
    return s

# Best layer per (HMM, parametrization) from the regular R² CSV: each representative's OWN best layer (as in Figure 3)
best_layers = {}
for hmm, param in TARGETS:
    sub = r2_df[(r2_df['hmm'] == hmm) & (r2_df['param'] == param) & (r2_df['target'] == 'real')]
    best_layers[(hmm, param)] = int(sub.groupby('layer')['R2'].mean().idxmax())

# match the 0-HMM / 1-HMM colors from the kl figure (RdYlGn_r ramp at k=1 and k=0)
_ramp = plt.get_cmap('RdYlGn_r')
_KMAX = float(kl_k['k'].max()) if 'kl_k' in globals() else 20.0
def _kcolor(k):
    return _ramp(min(max((k / _KMAX) ** 0.4, 0.0), 1.0))

dist_colors = {'real': 'black', 'order-1': _kcolor(1), 'order-0': _kcolor(0)}
dist_labels = {'real': 'Activations (Full HMM) → Beliefs (Full HMM)', 'order-1': 'Activations (1-HMM) → Beliefs (Full HMM)', 'order-0': 'Activations (0-HMM) → Beliefs (Full HMM)'}

fig, axes = plt.subplots(1, 4, figsize=(16, 3.3))

for i, (hmm, param) in enumerate(TARGETS):
    ax = axes[i]
    layer = best_layers[(hmm, param)]

    for dist_name in ['order-0', 'order-1', 'real']:
        s = redr2_df[(redr2_df['hmm'] == hmm) & (redr2_df['param'] == param) &
                      (redr2_df['dist'] == dist_name) & (redr2_df['layer'] == layer)]
        stats = s.groupby('k')['R2'].agg(['mean', 'sem']).reset_index()
        lw = 5
        ax.plot(stats['k'], stats['mean'], '-',
                color=dist_colors[dist_name], lw=lw, label=dist_labels[dist_name])
        ax.fill_between(stats['k'],
                        stats['mean'] - 1.96 * stats['sem'],
                        stats['mean'] + 1.96 * stats['sem'],
                        color=dist_colors[dist_name], alpha=0.15, linewidth=0)

    ax.set_title(f'{hmm} (L{layer})', fontsize=23, pad=18, y=0.95)
    ax.set_xticks([0, 5, 10, 15, 20])
    ax.set_xlabel('$k$ (suffix length)', fontsize=22)
    ax.tick_params(axis='both', labelsize=22, length=3)
    ax.grid(True, alpha=0.2, linewidth=0.5)
    if i == 0:
        ax.set_ylabel('R²', fontsize=22)

    ax.yaxis.set_major_locator(MaxNLocator(nbins=3, min_n_ticks=3))
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: fmt_tick(v)))

# Legend below — real alone in left column; order-0 (top) and order-1 (bottom) in right column
_spacer = Line2D([], [], color='none', label=' ')
legend_handles = [Line2D([], [], color=dist_colors['real'],    lw=5, label=dist_labels['real']),
                  _spacer,
                  Line2D([], [], color=dist_colors['order-0'], lw=5, label=dist_labels['order-0']),
                  Line2D([], [], color=dist_colors['order-1'], lw=5, label=dist_labels['order-1'])]
fig.legend(handles=legend_handles, loc='lower center', ncol=2,
           fontsize=22, frameon=False, bbox_to_anchor=(0.5, -0.325))

plt.tight_layout(w_pad=0)
plt.savefig(f'{PLOT_DIR}/redr2.pdf', bbox_inches='tight')
plt.show()
plt.close()

In [ ]:
K_ASYMP = 20

for model_key in MODEL_KEYS:
    model_name = MODEL_LABELS.get(model_key, model_key)
    redr2_df = load_csv('ksuffix_probes', model_key)
    if redr2_df is None:
        continue

    redr2_data = redr2_df[redr2_df['hmm'] != 'Spiral']
    hmms = [h for h in HMM_ORDER if h in redr2_data['hmm'].values]

    fig = plt.figure(figsize=(13, 8))
    fig.patch.set_facecolor('white')
    fig.patch.set_edgecolor('white')
    fig.patch.set_linewidth(0)
    outer = fig.add_gridspec(2, 2, wspace=0.55, hspace=0.3)

    all_axes = []
    for j, hmm in enumerate(hmms[:4]):
        r, c = divmod(j, 2)
        ax = fig.add_subplot(outer[r, c])
        all_axes.append((ax, hmm, r, c))

        hmm_data = redr2_data[redr2_data['hmm'] == hmm]
        params = sorted(hmm_data['param'].unique())
        n_p = len(params)
        tp = np.linspace(0.3, 0.9, n_p)
        colors = plt.cm.Greens(tp)

        for i, param in enumerate(params):
            sub = hmm_data[(hmm_data['param'] == param) & (hmm_data['k'] == K_ASYMP)]

            real = sub[sub['dist'] == 'real'].groupby(['layer', 'seed'])['R2'].first().reset_index()
            ord1 = sub[sub['dist'] == 'order-1'].groupby(['layer', 'seed'])['R2'].first().reset_index()

            merged = real.merge(ord1, on=['layer', 'seed'], suffixes=('_real', '_ord1'))
            merged['delta'] = merged['R2_real'] - merged['R2_ord1']

            stats = merged.groupby('layer')['delta'].agg(['mean', 'sem']).reset_index()

            ax.plot(stats['layer'], stats['mean'], '-', color=colors[i], lw=3,
                    label=param)
            ax.fill_between(stats['layer'],
                            stats['mean'] - 1.96 * stats['sem'],
                            stats['mean'] + 1.96 * stats['sem'],
                            color=colors[i], alpha=0.15, linewidth=0)

        ax.axhline(0, color='black', lw=3, ls='-', zorder=10)
        ax.set_title(hmm, fontsize=16)
        ax.tick_params(axis='both', labelsize=13, length=3)
        ax.grid(True, alpha=0.2, linewidth=0.5)
        ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: fmt_tick(v)))

        n_layers = int(hmm_data['layer'].max())
        ax.set_xlim(-0.5, n_layers + 0.5)
        ax.set_xticks(np.arange(0, n_layers + 1, 5))

        if r == 1: ax.set_xlabel('Layer', fontsize=14)

        # Legend to the right
        param_handles = [Line2D([], [], color=colors[i], lw=3, label=p)
                         for i, p in enumerate(params)]
        ax.legend(handles=param_handles, fontsize=9.5, ncol=1,
                  loc='center left', bbox_to_anchor=(1.02, 0.5),
                  frameon=True, edgecolor='0.8',
                  borderpad=0.4, handlelength=1.2, labelspacing=0.3)

    # ylabel centered per row
    fig.canvas.draw()
    for row_idx in range(2):
        row_axes = [ax for ax, _, r, c in all_axes if r == row_idx and c == 0]
        if not row_axes: continue
        ax = row_axes[0]
        pos = ax.get_position()
        fig.text(pos.x0 - 0.05, (pos.y0 + pos.y1) / 2,
                 '$\Delta$R² (HMM $-$ 1-HMM)', fontsize=15,
                 va='center', ha='center', rotation=90)

    fig.suptitle(model_name, fontsize=16, y=0.98)
    plt.savefig(f'{PLOT_DIR}/rep_gap_k20_{model_key}.pdf', bbox_inches='tight')
    plt.show(); plt.close()

In [ ]:
# ── HMM helpers for the theoretical counterparts (Appendices K and L) ──
import glob
sys.path.insert(0, '..')
from src.hmm import stationary_distribution, sample_hmm_sequence, full_bayesian_beliefs, next_token_probs
from src.hmm.core import precompute_belief_tables, compute_k_beliefs
from src.hmm.definitions import (mess3_matrices, arch_matrices, wing_matrices, strata_matrices,
                                 mess3_order_one, arch_order_one, wing_order_one, strata_order_one)
BUILDERS    = {'Mess3': (mess3_matrices, ['a', 'x']), 'Arch': (arch_matrices, ['a']),
               'Wing': (wing_matrices, ['a', 'x']), 'Strata': (strata_matrices, ['a', 't0', 't1'])}
BUILDERS_O1 = {'Mess3': mess3_order_one, 'Arch': arch_order_one, 'Wing': wing_order_one, 'Strata': strata_order_one}

def parse_param(label):
    """'a=0.97, t0=0.38, t1=0.54' -> {'a': 0.97, 't0': 0.38, 't1': 0.54}"""
    return {k: float(v) for k, v in (kv.split('=') for kv in label.split(', '))}

def ksuffix_beliefs(tokens, T_stack, pi, k):
    """Belief from the stationary prior over the k tokens ending at position t, for t = k, ..., N-1
    (element j <-> position j + k, window tokens[j+1 .. j+k])."""
    tokens = np.asarray(tokens, dtype=np.int64); n_tok = T_stack.shape[0]; N = len(tokens)
    max_k_lookup = 20 if n_tok == 2 else 12
    tables = precompute_belief_tables([k], list(T_stack), pi) if k <= max_k_lookup else {}
    return compute_k_beliefs(tokens, k, N - k, [k], tables, T_stack, pi, n_tok, max_k_lookup)[k]


In [ ]:
res = redr2_df[redr2_df['hmm'].isin(HMM_ORDER)][['hmm', 'param']].drop_duplicates()   # the (family, parametrization) pairs
# ══ seed-level 1-HMM theory gap + matched ΔR² — per model, 1×4, 100 points ══
#   tgap_seed: on the TRUE-process seed-s sequence, 1 − R²( true k=20 suffix beliefs ~ 1-HMM k=20 beliefs ),
#              probe window 15000:, split train_test_split(0.2, random_state=seed), bias, test-set R².
#   ΔR²_seed:  R²(real, seed) − R²(order-1, seed) at k=20, best layer (chosen on real-arm seed-mean).
from sklearn.model_selection import train_test_split
SEQ_LEN, PROBE_START, TRAIN_FRAC, N_SEEDS, KWIN = 20000, 15000, 0.2, 10, 20
CACHE1 = f'{RESULTS_DIR}/theory_gap_1hmm_seedlevel.csv'

if os.path.exists(CACHE1):
    tg1_seed = pd.read_csv(CACHE1)
else:
    rows = []
    for (hmm, param) in res[['hmm','param']].drop_duplicates().itertuples(index=False):
        fn, keys = BUILDERS[hmm]; d = parse_param(param)
        T  = fn(*[d[k] for k in keys]);  pi  = stationary_distribution(T);  T_stack  = np.stack(T)
        T1 = BUILDERS_O1[hmm](*[d[k] for k in keys]); pi1 = stationary_distribution(T1); T1_stack = np.stack(T1)
        for seed in range(N_SEEDS):
            toks = sample_hmm_sequence(T, pi, SEQ_LEN, seed=seed)
            Y = ksuffix_beliefs(toks, T_stack,  pi,  KWIN)[PROBE_START-KWIN:]   # true k=20 suffix beliefs
            X = ksuffix_beliefs(toks, T1_stack, pi1, KWIN)[PROBE_START-KWIN:]   # 1-HMM k=20 beliefs, same tokens
            n = len(Y)
            tr, te = train_test_split(np.arange(n), train_size=TRAIN_FRAC, random_state=seed)
            Xb = np.hstack([X, np.ones((n,1))])
            W = np.linalg.pinv(Xb[tr]) @ Y[tr]
            r2 = 1 - ((Y[te]-Xb[te]@W)**2).sum() / ((Y[te]-Y[te].mean(0))**2).sum()
            rows.append(dict(hmm=hmm, param=param, seed=seed, tgap_seed=1-r2))
        print(f'  done {hmm} {param}')
    tg1_seed = pd.DataFrame(rows); tg1_seed.to_csv(CACHE1, index=False)

# per-seed ΔR² from redr2 files
rows = []
for f in sorted(glob.glob(f'{RESULTS_DIR}/ksuffix_probes_*.csv')):
    mk = os.path.basename(f)[len('ksuffix_probes_'):-4]
    df = pd.read_csv(f); df = df[df.hmm != 'Spiral']
    for (hmm, param), g in df.groupby(['hmm','param']):
        kmax = g.k.max(); gk = g[g.k == kmax]
        L = gk[gk.dist=='real'].groupby('layer')['R2'].mean().idxmax()
        real = gk[(gk.dist=='real')   & (gk.layer==L)].set_index('seed')['R2']
        o1   = gk[(gk.dist=='order-1')& (gk.layer==L)].set_index('seed')['R2']
        dd = (real - o1).dropna()
        for seed, v in dd.items():
            rows.append(dict(model=mk, hmm=hmm, param=param, seed=seed, dR2=v))
res_seed = pd.DataFrame(rows).merge(tg1_seed, on=['hmm','param','seed'], how='left')

# 1×4 per model — ΔR² vs theoretical counterpart
for mk in MODEL_ORDER:
    sm = res_seed[res_seed.model == mk]
    if len(sm) == 0: continue
    fig, axes = plt.subplots(1, 4, figsize=(16, 3.6))
    print(f"{MODEL_LABELS.get(mk, mk)} — ΔR² vs theoretical counterpart (seed-level)")
    for i, (ax, hmm) in enumerate(zip(axes, HMMS)):
        s = sm[sm.hmm == hmm].dropna(subset=['tgap_seed'])
        ax.scatter(s.tgap_seed, s.dR2, color=HMM_COLORS[hmm], s=26, alpha=0.6,
                   edgecolor='k', linewidth=0.2)
        pr, _ = pearsonr(s.tgap_seed, s.dR2); sr, _ = spearmanr(s.tgap_seed, s.dR2)
        print(f"    {hmm:7s} pearson {pr:+.2f}   spearman {sr:+.2f}   n={len(s)}")
        ax.set_title(f'{hmm}  ($r$={pr:.2f})', fontsize=17)
        ax.set_xlabel('$1-R^2$ (1-HMM $\\to$ true beliefs, $k$=20)', fontsize=14)
        ax.grid(alpha=0.25); ax.tick_params(labelsize=12)
        if i == 0:
            ax.set_ylabel('$\\Delta R^2$ rel. 1-HMM ($k$=20)', fontsize=14)
    fig.suptitle(f'{MODEL_LABELS.get(mk, mk)}', fontsize=16, y=1.0)
    plt.tight_layout()
    plt.savefig(f'{PLOT_DIR}/rep_gap_theory_seed_{mk}.pdf', bbox_inches='tight')
    plt.show(); plt.close()

## Figure 5 (bottom) and Appendix L: NTP and log-NTP probes and baselines (Figures 47 to 52)

In [ ]:
# Load the NTP and log-NTP probe results of the main-text model (Figure 5, bottom).
MODEL = 'qwen35_9b'
r2_df = load_csv('r2', MODEL)
obsprob_df = load_csv('ntp_probes', MODEL)
print(f'Obsprob: {len(obsprob_df)} rows') if obsprob_df is not None else None


In [ ]:
import matplotlib
from matplotlib.ticker import MaxNLocator
matplotlib.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['DejaVu Sans'],
    'font.weight': 'light',
})
sns.set_context('notebook')

def fmt_tick(v):
    if v == int(v):
        s = f'{v:.1f}'
    else:
        s = f'{v:.2f}'
    if '.' in s and -1 < v < 1:
        s = s.lstrip('0')
    return s

_purple = tab10[4]   # purple
_pink   = tab10[6]   # pink
target_styles = {
    'real':            ('black',  5, '-',  'Activations → Beliefs'),
    'act→ntp':         (_purple,  3, '--', "Activations → NTP"),
    'act→log_ntp':     (_pink,    3, '--', "Activations → log-NTP"),
    'ntp→beliefs':     (_purple,  5, '-',  "NTP → Beliefs"),
    'log_ntp→beliefs': (_pink,    5, '-',  "log-NTP → Beliefs"),
}

targets_all = list(TARGETS)                      # include Mess3
fig, axes = plt.subplots(1, len(targets_all), figsize=(16, 3.3))   # match redr2

for i, (hmm, param) in enumerate(targets_all):
    ax = axes[i]

    for tname in ['act→log_ntp', 'log_ntp→beliefs', 'act→ntp', 'ntp→beliefs']:
        color, lw, ls, label = target_styles[tname]
        s = obsprob_df[(obsprob_df['hmm'] == hmm) & (obsprob_df['param'] == param) &
                        (obsprob_df['target'] == tname)]
        stats = s.groupby('layer')['R2'].agg(['mean', 'sem']).reset_index()
        ax.plot(stats['layer'], stats['mean'], ls=ls, color=color, lw=lw)
        ax.fill_between(stats['layer'],
                        stats['mean'] - 1.96 * stats['sem'],
                        stats['mean'] + 1.96 * stats['sem'],
                        color=color, alpha=0.1, linewidth=0)

    # Real R² on top
    r2_sub = r2_df[(r2_df['hmm'] == hmm) & (r2_df['param'] == param) & (r2_df['target'] == 'real')]
    stats = r2_sub.groupby('layer')['R2'].agg(['mean', 'sem']).reset_index()
    ax.plot(stats['layer'], stats['mean'], color='black', lw=5)
    ax.fill_between(stats['layer'],
                    stats['mean'] - 1.96 * stats['sem'],
                    stats['mean'] + 1.96 * stats['sem'],
                    color='black', alpha=0.15, linewidth=0)

    ax.set_title(f'{hmm}', fontsize=23, pad=18, y=0.95)
    ax.set_xlabel('Layer', fontsize=22)
    ax.set_xticks([0, 10, 20, 30])
    ax.tick_params(axis='both', labelsize=22, length=3)
    ax.grid(True, alpha=0.2, linewidth=0.5)
    ax.yaxis.set_major_locator(MaxNLocator(nbins=3, min_n_ticks=3))
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: fmt_tick(v)))
    if i == 0:
        ax.set_ylabel('R²', fontsize=22)

# Legend
legend_order = ['real', None, 'ntp→beliefs', 'log_ntp→beliefs', 'act→ntp', 'act→log_ntp']
legend_handles, legend_labels = [], []
for tname in legend_order:
    if tname is None:
        legend_handles.append(Line2D([], [], lw=0, alpha=0)); legend_labels.append('')
    else:
        color, lw, ls, label = target_styles[tname]
        legend_handles.append(Line2D([], [], color=color, lw=5, ls=ls)); legend_labels.append(label)
fig.legend(handles=legend_handles, labels=legend_labels, loc='lower center', ncol=3,
           fontsize=22, frameon=False, bbox_to_anchor=(0.5, -0.325))   # match redr2

plt.tight_layout(w_pad=0)
plt.savefig(f'{PLOT_DIR}/obsprob.pdf', bbox_inches='tight')
plt.show(); plt.close()

In [ ]:
# ΔR² = R²(act→beliefs) - R²(NTP→beliefs), ALL families (incl. Mess3)
for model_key in MODEL_KEYS:
    model_name = MODEL_LABELS.get(model_key, model_key)
    obsprob_df = load_csv('ntp_probes', model_key)
    r2_df_model = load_csv('r2', model_key)
    if obsprob_df is None or r2_df_model is None:
        continue

    hmms = [h for h in HMM_ORDER if h in obsprob_df['hmm'].values]   # incl. Mess3

    fig = plt.figure(figsize=(13, 8))
    fig.patch.set_facecolor('white'); fig.patch.set_edgecolor('white'); fig.patch.set_linewidth(0)
    outer = fig.add_gridspec(2, 2, wspace=0.55, hspace=0.3)
    axes_list = [fig.add_subplot(outer[r, c]) for r in range(2) for c in range(2)]

    for j, hmm in enumerate(hmms[:4]):
        ax = axes_list[j]
        hmm_obs = obsprob_df[obsprob_df['hmm'] == hmm]
        params = sorted(hmm_obs['param'].unique())
        tp = np.linspace(0.3, 0.9, len(params))
        colors = plt.cm.Purples(tp)

        for i, param in enumerate(params):
            act_sub = r2_df_model[(r2_df_model['hmm'] == hmm) &
                                   (r2_df_model['param'] == param) &
                                   (r2_df_model['target'] == 'real')]
            act = act_sub.groupby(['layer', 'seed'])['R2'].first().reset_index()
            ntp_sub = hmm_obs[(hmm_obs['param'] == param) &
                               (hmm_obs['target'] == 'ntp→beliefs')]
            ntp = ntp_sub.groupby(['layer', 'seed'])['R2'].first().reset_index()
            merged = act.merge(ntp, on=['layer', 'seed'], suffixes=('_act', '_ntp'))
            merged['delta'] = merged['R2_act'] - merged['R2_ntp']
            stats = merged.groupby('layer')['delta'].agg(['mean', 'sem']).reset_index()
            ax.plot(stats['layer'], stats['mean'], '-', color=colors[i], lw=3, label=param)
            ax.fill_between(stats['layer'], stats['mean'] - 1.96 * stats['sem'],
                            stats['mean'] + 1.96 * stats['sem'],
                            color=colors[i], alpha=0.15, linewidth=0)

        ax.axhline(0, color='black', lw=4, ls='-', zorder=10)
        ax.set_title(hmm, fontsize=16)
        ax.set_xlabel('Layer', fontsize=14)
        ax.set_xticks([0, 10, 20, 30])
        ax.tick_params(axis='both', labelsize=13, length=3)
        ax.grid(True, alpha=0.2, linewidth=0.5)
        ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: fmt_tick(v)))
        ax.set_xlim(-0.5, int(hmm_obs['layer'].max()) + 0.5)

        param_handles = [Line2D([], [], color=colors[i], lw=3, label=p)
                         for i, p in enumerate(params)]
        ax.legend(handles=param_handles, fontsize=9.5, ncol=1, loc='center left',
                  bbox_to_anchor=(1.02, 0.5), frameon=True, edgecolor='0.8',
                  borderpad=0.4, handlelength=1.2, labelspacing=0.3)

    # ylabel on the two left-column panels
    fig.canvas.draw()
    for ax in (axes_list[0], axes_list[2]):
        pos = ax.get_position()
        fig.text(pos.x0 - 0.05, (pos.y0 + pos.y1) / 2,
                 '$\\Delta$R² (Act→Bel $-$ NTP→Bel)', fontsize=15,
                 va='center', ha='center', rotation=90)

    fig.suptitle(model_name, fontsize=16, y=0.98)
    plt.savefig(f'{PLOT_DIR}/ntp_gap_{model_key}.pdf', bbox_inches='tight')
    plt.show(); plt.close()

In [ ]:
# ΔR² = R²(act→beliefs) - R²(logNTP→beliefs), ALL families (incl. Mess3)
from matplotlib.colors import LinearSegmentedColormap

for model_key in MODEL_KEYS:
    model_name = MODEL_LABELS.get(model_key, model_key)
    obsprob_df = load_csv('ntp_probes', model_key)
    r2_df_model = load_csv('r2', model_key)
    if obsprob_df is None or r2_df_model is None:
        continue

    hmms = [h for h in HMM_ORDER if h in obsprob_df['hmm'].values]   # incl. Mess3

    fig = plt.figure(figsize=(13, 8))
    fig.patch.set_facecolor('white'); fig.patch.set_edgecolor('white'); fig.patch.set_linewidth(0)
    outer = fig.add_gridspec(2, 2, wspace=0.55, hspace=0.3)
    axes_list = [fig.add_subplot(outer[r, c]) for r in range(2) for c in range(2)]

    pinks = LinearSegmentedColormap.from_list('pinks', ['#fde4ef', '#f8a8cd', '#ef72ac'])

    for j, hmm in enumerate(hmms[:4]):
        ax = axes_list[j]
        hmm_obs = obsprob_df[obsprob_df['hmm'] == hmm]
        params = sorted(hmm_obs['param'].unique())
        colors = pinks(np.linspace(0.3, 0.9, len(params)))

        for i, param in enumerate(params):
            act_sub = r2_df_model[(r2_df_model['hmm'] == hmm) &
                                   (r2_df_model['param'] == param) &
                                   (r2_df_model['target'] == 'real')]
            act = act_sub.groupby(['layer', 'seed'])['R2'].first().reset_index()
            ntp_sub = hmm_obs[(hmm_obs['param'] == param) &
                               (hmm_obs['target'] == 'log_ntp→beliefs')]
            ntp = ntp_sub.groupby(['layer', 'seed'])['R2'].first().reset_index()
            merged = act.merge(ntp, on=['layer', 'seed'], suffixes=('_act', '_ntp'))
            merged['delta'] = merged['R2_act'] - merged['R2_ntp']
            stats = merged.groupby('layer')['delta'].agg(['mean', 'sem']).reset_index()
            ax.plot(stats['layer'], stats['mean'], '-', color=colors[i], lw=3, label=param)
            ax.fill_between(stats['layer'], stats['mean'] - 1.96 * stats['sem'],
                            stats['mean'] + 1.96 * stats['sem'],
                            color=colors[i], alpha=0.15, linewidth=0)

        ax.axhline(0, color='black', lw=4, ls='-', zorder=10)
        ax.set_title(hmm, fontsize=16)
        ax.set_xlabel('Layer', fontsize=14)
        ax.set_xticks([0, 10, 20, 30])
        ax.tick_params(axis='both', labelsize=13, length=3)
        ax.grid(True, alpha=0.2, linewidth=0.5)
        ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: fmt_tick(v)))
        ax.set_xlim(-0.5, int(hmm_obs['layer'].max()) + 0.5)

        param_handles = [Line2D([], [], color=colors[i], lw=3, label=p)
                         for i, p in enumerate(params)]
        ax.legend(handles=param_handles, fontsize=9.5, ncol=1, loc='center left',
                  bbox_to_anchor=(1.02, 0.5), frameon=True, edgecolor='0.8',
                  borderpad=0.4, handlelength=1.2, labelspacing=0.3)

    # ylabel on the two left-column panels
    fig.canvas.draw()
    for ax in (axes_list[0], axes_list[2]):
        pos = ax.get_position()
        fig.text(pos.x0 - 0.05, (pos.y0 + pos.y1) / 2,
                 '$\\Delta$R² (Act→Bel $-$ logNTP→Bel)', fontsize=15,
                 va='center', ha='center', rotation=90)

    fig.suptitle(model_name, fontsize=16, y=0.98)
    plt.savefig(f'{PLOT_DIR}/log_ntp_gap_{model_key}.pdf', bbox_inches='tight')
    plt.show(); plt.close()

In [ ]:
exc = obsprob_df[obsprob_df['hmm'].isin(HMM_ORDER)][['hmm', 'param']].drop_duplicates()   # the (family, parametrization) pairs
# ══ seed-level MATCHED NTP/log-NTP gaps: 100 points per panel ══
#   For each (hmm, param, seed): regenerate the exact sequence, reproduce the probe protocol
#   (window 15000:, split random_state=seed, train_frac=0.2, bias, R² on test), and compute
#   tgap_seed = 1 − R²(NTP or log-NTP → beliefs). excess_seed = R²(act→beliefs, best layer)
#   − R²(surrogate→beliefs). NOTE: ~400 sequence regenerations ≈ 10–20 min; cached to CSV.
from sklearn.model_selection import train_test_split
SEQ_LEN, PROBE_START, TRAIN_FRAC, N_SEEDS = 20000, 15000, 0.2, 10
CACHE = f'{RESULTS_DIR}/theory_gap_seedlevel.csv'

if os.path.exists(CACHE):
    tg_seed = pd.read_csv(CACHE)
else:
    rows = []
    for (hmm, param) in exc[['hmm','param']].drop_duplicates().itertuples(index=False):
        fn, keys = BUILDERS[hmm]; d = parse_param(param)
        T = fn(*[d[k] for k in keys]); pi = stationary_distribution(T); T_stack = np.stack(T)
        for seed in range(N_SEEDS):
            toks = sample_hmm_sequence(T, pi, SEQ_LEN, seed=seed)
            B = full_bayesian_beliefs(toks.astype(np.int64), T_stack, pi)
            ntp = next_token_probs(B, T)
            y  = B[PROBE_START:]; n_late = len(y)
            tr, te = train_test_split(np.arange(n_late), train_size=TRAIN_FRAC, random_state=seed)
            for surr, X in [('ntp', ntp[PROBE_START:]), ('log_ntp', np.log(ntp[PROBE_START:] + 1e-12))]:
                Xb = np.hstack([X, np.ones((len(X), 1))])
                W = np.linalg.pinv(Xb[tr]) @ y[tr]
                pred = Xb[te] @ W
                r2 = 1 - ((y[te]-pred)**2).sum() / ((y[te]-y[te].mean(0))**2).sum()
                rows.append(dict(hmm=hmm, param=param, seed=seed, surrogate=surr,
                                 ceil_seed=r2, tgap_seed=1-r2))
        print(f'  done {hmm} {param}')
    tg_seed = pd.DataFrame(rows); tg_seed.to_csv(CACHE, index=False)

# merge with per-seed act→beliefs (best layer per model×hmm×param, chosen on the seed-mean)
rows = []
for f in sorted(glob.glob(f'{RESULTS_DIR}/ntp_probes_*.csv')):
    mk = os.path.basename(f)[len('ntp_probes_'):-4]
    df = pd.read_csv(f); df = df[df.hmm != 'Spiral']
    for (hmm, param), g in df.groupby(['hmm', 'param']):
        gb = g[g.target == 'act→beliefs'].groupby('layer')['R2'].mean()
        if len(gb) == 0: continue
        L = gb.idxmax()
        actb = g[(g.target == 'act→beliefs') & (g.layer == L)].set_index('seed')['R2']
        for seed, a in actb.items():
            for surr in ['ntp', 'log_ntp']:
                t = tg_seed[(tg_seed.hmm==hmm)&(tg_seed.param==param)&
                            (tg_seed.seed==seed)&(tg_seed.surrogate==surr)]
                if len(t) == 0: continue
                rows.append(dict(model=mk, hmm=hmm, param=param, seed=seed, surrogate=surr,
                                 tgap=float(t.tgap_seed.iloc[0]),
                                 excess=a - float(t.ceil_seed.iloc[0])))
exc_seed = pd.DataFrame(rows)

# per-model 1×4 figures, 100 matched points per panel
for surr, lab in [('ntp', 'NTP'), ('log_ntp', 'log-NTP')]:
    for mk in MODEL_ORDER:
        sm = exc_seed[(exc_seed.model == mk) & (exc_seed.surrogate == surr)]
        if len(sm) == 0: continue
        fig, axes = plt.subplots(1, 4, figsize=(16, 3.6))
        print(f"{MODEL_LABELS.get(mk, mk)} — {lab} (seed-level, matched)")
        for i, (ax, hmm) in enumerate(zip(axes, HMMS)):
            s = sm[sm.hmm == hmm]
            ax.scatter(s.tgap, s.excess, color=HMM_COLORS[hmm], s=26, alpha=0.6,
                       edgecolor='k', linewidth=0.2)
            if s.tgap.std() > 1e-9:
                pr, _ = pearsonr(s.tgap, s.excess); sr, _ = spearmanr(s.tgap, s.excess)
                title = f'{hmm}  ($r$={pr:.2f})'
                print(f"    {hmm:7s} pearson {pr:+.2f}   spearman {sr:+.2f}   n={len(s)}")
            else:
                title = f'{hmm}  (gap $\\approx$ 0)'
            lim = max(s.tgap.max(), s.excess.max(), 1e-3) * 1.1
            ax.plot([0, lim], [0, lim], 'k--', lw=1, alpha=0.5)
            ax.set_title(title, fontsize=17)
            ax.set_xlabel(f'$1-R^2$ ({lab} $\\to$ beliefs)', fontsize=14)
            ax.grid(alpha=0.25); ax.tick_params(labelsize=12)
            if i == 0:
                ax.set_ylabel(f'$R^2$ (act.$\\to$ bel.) $-$ $R^2$ ({lab} $\\to$ bel.)',
                              fontsize=13)
        fig.suptitle(f'{MODEL_LABELS.get(mk, mk)}', fontsize=16, y=1.0)
        plt.tight_layout()
        plt.savefig(f'{PLOT_DIR}/ntp_gap_theory_seed_{surr}_{mk}.pdf', bbox_inches='tight')
        plt.show(); plt.close()